# Audio Harness: GPU-сервер на Colab T4

Поднимает `separate` (Mel-Band RoFormer) и `restore` (Sidon) на GPU и открывает их через Cloudflare quick tunnel. Локальный `ah` на Mac ходит сюда при `backend = "remote"`.

Запускайте ячейки по порядку: Runtime → Change runtime type → **T4 GPU**.

Если туннель не нужен, используйте последний раздел: тот же `ah` прямо в Colab, без сети наружу.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Код
Либо `git clone` вашего репозитория, либо загрузите `audio-harness.zip`.

In [ ]:
import os, pathlib
REPO_URL = ""  # например https://github.com/<you>/audio-harness.git
if REPO_URL:
    !git clone --depth 1 {REPO_URL} /content/audio-harness
elif not pathlib.Path('/content/audio-harness').exists():
    from google.colab import files
    uploaded = files.upload()  # audio-harness.zip
    !unzip -q -o {next(iter(uploaded))} -d /content
%cd /content/audio-harness

## 2. Зависимости
Core ставится первым, чтобы инструменты нашли его локально, а не на PyPI. `analyze` по умолчанию остаётся на Mac (CPU, madmom собирается долго); раскомментируйте, если нужен и он.

In [ ]:
%pip install -q -e '.[server]' -e tools/separate -e tools/restore
# %pip install -q -e tools/analyze

## 3. Токен и сервер
Токен печатается один раз. Скопируйте его в `AH_REMOTE_TOKEN` на Mac.

In [ ]:
import secrets, subprocess, time, os
TOKEN = secrets.token_urlsafe(32)
env = dict(os.environ, AH_SERVER_TOKEN=TOKEN, AH_SERVER_TOOLS='separate,restore')
server = subprocess.Popen(['ah', 'serve', '--host', '127.0.0.1', '--port', '8765'], env=env,
                          stdout=open('/content/server.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(8)
print('AH_REMOTE_TOKEN =', TOKEN)
!tail -3 /content/server.log

## 4. Туннель
Quick tunnel даёт случайный `*.trycloudflare.com`. Сервер всё равно требует токен; без токена любой запрос получает 401.

In [ ]:
CLOUDFLARED = '2024.12.2'  # закреплённая версия; обновляйте осознанно
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/download/{CLOUDFLARED}/cloudflared-linux-amd64 && chmod +x /content/cloudflared
import re
tunnel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8765'],
                          stdout=open('/content/tunnel.log', 'w'), stderr=subprocess.STDOUT)
URL = None
for _ in range(30):
    time.sleep(2)
    found = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/tunnel.log').read())
    if found:
        URL = found.group(0); break
print('AH_REMOTE_URL =', URL)

In [ ]:
import httpx
print(httpx.get(f'{URL}/v1/health', headers={'Authorization': f'Bearer {TOKEN}'}, timeout=30).json())

## 5. Прогрев
Первый вызов скачивает веса (Kim ~913 МБ, Sidon ~1 ГБ + w2v-BERT). Прогреваем на тестовом сигнале, чтобы первый запрос с Mac не упёрся в таймаут.

In [ ]:
import numpy as np, soundfile as sf
sr = 44100; t = np.arange(sr * 5) / sr
sf.write('/content/warmup.wav', np.stack([0.3 * np.sin(2 * np.pi * 220 * t)] * 2, 1).astype('float32'), sr)
for tool in ('separate', 'restore'):
    with open('/content/warmup.wav', 'rb') as f:
        r = httpx.post(f'{URL}/v1/tools/{tool}', headers={'Authorization': f'Bearer {TOKEN}'},
                       files={'audio': ('warmup.wav', f, 'audio/wav')}, data={'params': '{}'}, timeout=1800)
    print(tool, r.status_code, len(r.content), 'bytes')

## Без туннеля: `ah` прямо в Colab
Всё то же самое, но файлы живут в `/content/ah`, а инструменты работают в этом же процессе.

In [ ]:
%env AH_HOME=/content/ah
!ah import /content/warmup.wav --label warmup
!ah separate --backend inprocess
!ah tree